## Aplicación de decisiones del EDA

En el Análisis Exploratorio de Datos (EDA) se identificaron una serie de conclusiones relevantes
que guían el proceso de ingeniería de variables:

- El dataset no presenta valores nulos, por lo que no es necesaria la imputación de valores faltantes.
- Se detectó un único registro completamente duplicado, que se elimina para evitar sesgos en el modelado.
- Existen valores extremos en la variable `charges`, pero se consideran plausibles y representativos,
  por lo que no se eliminan.
- La variable `age` se agrupa en intervalos de edad y se sustituye por el valor medio del intervalo
  al que pertenece cada individuo, con el objetivo de suavizar la variable y reducir ruido.
- Las variables categóricas (`sex`, `smoker`, `region`) requieren codificación.
- Las variables numéricas presentan escalas distintas, por lo que se aplicará escalado.
- La variable objetivo del proyecto es `charges`, por lo que el problema se plantea como
  un problema de regresión supervisada.

A continuación, se aplican estas decisiones de forma sistemática y reproducible.

### Importamos librerias

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

### Leemos el dataset 

In [2]:
df = pd.read_csv("../data/insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


### Empezamos la limpieza

In [3]:
df = df.drop_duplicates()

In [4]:
bins = [18, 25, 35, 45, 55, 100]
labels = ["18-25", "26-35", "36-45", "46-55", "56+"]

df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels, right=True)

Cogemos la edad media de cada intervalo 

In [5]:
interval_means = {
    "18-25": (18 + 25) / 2,
    "26-35": (26 + 35) / 2,
    "36-45": (36 + 45) / 2,
    "46-55": (46 + 55) / 2,
    "56+": 56 + 10  # Lo hacemos para conservar un poco a los de más edad
}

df["age_group_mean"] = df["age_group"].map(interval_means)


### TRAIN / TEST

Seleccionamos: X e y 

In [6]:
TARGET = "charges"

X = df.drop(columns=[TARGET, "age", "age_group"])
y = df[TARGET]

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

### Identificación de variables numéricas y categóricas

En esta sección se separan las variables del conjunto de entrenamiento según su tipo de dato.
Las variables numéricas se utilizarán para el escalado, mientras que las variables categóricas
se transformarán mediante codificación *one-hot*.

Esta separación es necesaria para aplicar distintos preprocesamientos a cada tipo de variable
dentro de un mismo pipeline.


In [8]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

num_cols, cat_cols

(Index(['bmi', 'children'], dtype='object'),
 Index(['sex', 'smoker', 'region'], dtype='object'))

### Definición del preprocesamiento (numérico y categórico)

En esta etapa se construye el flujo de preprocesamiento que se aplicará a las variables antes de entrenar el modelo.

- **Variables numéricas:** se escalan mediante `StandardScaler`, con el objetivo de estandarizar sus valores (media 0 y desviación estándar 1) y hacerlas comparables entre sí.

- **Variables categóricas:** se transforman mediante `OneHotEncoder`, convirtiendo cada categoría en variables binarias. Se utiliza `drop="first"` para evitar multicolinealidad y `handle_unknown="ignore"` para prevenir errores si aparecen categorías nuevas en los datos de validación o test.

Finalmente, `ColumnTransformer` integra ambos procesos para aplicar automáticamente el tratamiento adecuado a cada conjunto de columnas dentro de un único preprocesador.


In [9]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

### Continuación del análisis: preparación de los datos

Después de hacer el análisis exploratorio (EDA) y conocer bien el dataset, el siguiente paso fue preparar los datos para poder entrenar el modelo de forma correcta. En esta parte nos centramos en limpiar los datos y en aplicar las transformaciones necesarias antes del modelado.

Primero, se eliminaron las posibles filas duplicadas para evitar que la información repetida afectara al entrenamiento del modelo.

Luego se trabajó con la variable edad, que en el EDA vimos que era importante, pero no necesariamente lineal. Para ello, se agrupó la edad en intervalos y se sustituyó por la edad media de cada grupo, creando una nueva variable numérica. De esta forma se mantiene la información de la edad, pero de una manera más sencilla para el modelo.

Después, se definió charges como la variable que queremos predecir y se eliminaron la edad original y el grupo de edad categórico para no repetir información.

Para poder evaluar bien el modelo, los datos se dividieron en tres partes: entrenamiento, validación y test. Esto permite entrenar el modelo, ajustarlo y comprobar su rendimiento final con datos que no ha visto antes.

A continuación, se separaron las variables según su tipo. Las variables numéricas (bmi y children) se estandarizaron para que estuvieran en la misma escala. Las variables categóricas (sex, smoker y region) se transformaron con codificación one-hot, convirtiéndolas en variables binarias para que el modelo pueda trabajar correctamente con ellas.

Todo este preprocesamiento se integró en un único pipeline usando ColumnTransformer, lo que permite aplicar las mismas transformaciones a los datos de entrenamiento, validación y test, evitando errores y asegurando que el proceso sea consistente.

En resumen, esta fase deja los datos listos para entrenar modelos de forma más fiable, siguiendo lo observado en el EDA y asegurando una evaluación correcta del rendimiento del modelo